# Kuppai-to-Kaasu: Waste classifier v2 (target 90%+)
Run on **Kaggle** with GPU T4 and Internet ON. Takes about 25-40 minutes.

What changed from v1 (80.5% test accuracy):
- Drops the noisy TrashNet "trash" class (only 137 images, F1 0.30). Unclear items are handled in the app with a confidence threshold instead.
- Larger input (256x256) and a much longer, fuller fine-tune (all layers except BatchNorm).
- Stronger augmentation, label smoothing, learning-rate decay, early stopping with best weights.
- Checks the TFLite file's accuracy, not just the Keras model.

## 0. Settings

In [ ]:
USE_OTHER = False          # False = drop TrashNet "trash" class (recommended)
BACKBONE = "mobilenetv2"   # or "efficientnetv2b0" if you still want more accuracy
IMG = (256, 256)
BATCH = 32

## 1. Download TrashNet (resized, about 43 MB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os, glob, shutil

zip_path = hf_hub_download(repo_id="garythung/trashnet", filename="dataset-resized.zip", repo_type="dataset")
os.makedirs("/kaggle/working/raw", exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/kaggle/working/raw")

## 2. Merge classes and split 70 / 15 / 15

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

MAP = {"cardboard": "paper_cardboard", "paper": "paper_cardboard",
       "plastic": "plastic", "metal": "metal", "glass": "glass"}
if USE_OTHER:
    MAP["trash"] = "other"

base = None
for root, dirs, files in os.walk("/kaggle/working/raw"):
    if "__MACOSX" in root:
        continue
    if {"cardboard", "paper", "plastic", "metal", "glass"}.issubset(set(dirs)):
        base = root
        break
assert base, "Could not find the class folders."
print("Using", base)

items = []
for src, dst in MAP.items():
    for p in glob.glob(f"{base}/{src}/*"):
        if p.lower().endswith((".jpg", ".jpeg", ".png")):
            items.append((p, dst))
print("TrashNet images:", len(items))

## 3. (Recommended) Add your own photos
Real phone photos matter more than anything else for real-world accuracy, because TrashNet photos are taken on a plain white background.

Make `extra.zip` with folders named exactly `ewaste`, `organic`, and optionally `plastic`, `metal`, `glass`, `paper_cardboard` (photos of real items at home, on tables, floors, in bins). 60-100 photos per class. Upload it to the Colab Files panel as `EXTRA_ZIP`, then run the cell.

In [ ]:
EXTRA = glob.glob("/kaggle/input/**/extra.zip", recursive=True)
EXTRA_ZIP = EXTRA[0] if EXTRA else "/kaggle/working/extra.zip"
VALID = {"ewaste", "organic", "plastic", "metal", "glass", "paper_cardboard", "other"}
if os.path.exists("EXTRA_ZIP"):
    with zipfile.ZipFile("EXTRA_ZIP") as z:
        z.extractall("/kaggle/working/extra")
    for p in glob.glob("/kaggle/working/extra/**/*", recursive=True):
        if p.lower().endswith((".jpg", ".jpeg", ".png")):
            label = os.path.basename(os.path.dirname(p))
            if label in VALID:
                items.append((p, label))
    print("Added your photos.")
else:
    print("No extra.zip found. Training on TrashNet classes only.")
print(Counter(l for _, l in items))

In [ ]:
paths = [p for p, _ in items]
labels = [l for _, l in items]

p_train, p_tmp, l_train, l_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=42)
p_val, p_test, l_val, l_test = train_test_split(p_tmp, l_tmp, test_size=0.50, stratify=l_tmp, random_state=42)

shutil.rmtree("/kaggle/working/data", ignore_errors=True)
for split, ps, ls in [("train", p_train, l_train), ("val", p_val, l_val), ("test", p_test, l_test)]:
    for i, (p, l) in enumerate(zip(ps, ls)):
        d = f"/kaggle/working/data/{split}/{l}"
        os.makedirs(d, exist_ok=True)
        shutil.copy(p, f"{d}/{i}_{os.path.basename(p)}")
print(len(p_train), len(p_val), len(p_test))

## 4. Load the data

In [ ]:
import tensorflow as tf

def load(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"/kaggle/working/data/{split}", image_size=IMG, batch_size=BATCH,
        label_mode="categorical", shuffle=shuffle, seed=42)

train_ds = load("train", True)
val_ds = load("val", False)
test_ds = load("test", False)
class_names = train_ds.class_names
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## 5. Build the model

In [ ]:
from tensorflow.keras import layers, models

aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom((-0.25, 0.1)),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
    layers.RandomContrast(0.25),
])

if BACKBONE == "efficientnetv2b0":
    base_model = tf.keras.applications.EfficientNetV2B0(input_shape=IMG + (3,), include_top=False, weights="imagenet")
    scale = lambda t: t                      # preprocessing is built in (expects 0-255)
else:
    base_model = tf.keras.applications.MobileNetV2(input_shape=IMG + (3,), include_top=False, weights="imagenet")
    scale = layers.Rescaling(1/127.5, offset=-1)

base_model.trainable = False

inputs = layers.Input(shape=IMG + (3,))      # raw pixels 0-255
x = aug(inputs)
x = scale(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = models.Model(inputs, outputs)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10)

## 6. Fine-tune the whole backbone (BatchNorm stays frozen)

In [ ]:
base_model.trainable = True
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss, metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
]
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

## 7. Test accuracy (write this in the README)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

loss_v, acc = model.evaluate(test_ds)
print("KERAS TEST ACCURACY:", round(acc * 100, 2), "%")

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

## 8. Export to TFLite (float16) and verify the TFLite accuracy

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open("waste_classifier.tflite", "wb") as f:
    f.write(tflite_model)
with open("labels.txt", "w") as f:
    f.write("\n".join(class_names))
print("Size (MB):", round(len(tflite_model) / 1e6, 2))
print("Labels:", class_names)

interp = tf.lite.Interpreter(model_path="waste_classifier.tflite")
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Input:", inp["shape"], inp["dtype"], "Output:", out["shape"])

correct = total = 0
for xb, yb in test_ds:
    for img, y in zip(xb.numpy(), yb.numpy()):
        interp.set_tensor(inp["index"], img[None].astype(np.float32))
        interp.invoke()
        pred = np.argmax(interp.get_tensor(out["index"])[0])
        correct += int(pred == np.argmax(y))
        total += 1
print("TFLITE TEST ACCURACY:", round(100 * correct / total, 2), "%")

print("Saved in /kaggle/working. After the run, download both files from the Output tab (right panel).")

## Notes for the app
- Input: 256 x 256 x 3, float32, raw pixels 0-255. Scaling is inside the model, so do not normalise again.
- Output: one probability per class, in the order of `labels.txt`.
- If the top probability is below about 0.60, show "Not sure, try another angle" or treat it as "other" instead of guessing.
- Test accuracy on TrashNet is higher than real-world accuracy. Check about 10 real items with the phone camera and report both numbers.